# CredResolve Collections Analysis

**Goal:** check whether the reported 11% month-on-month recovery improvement is supported by the raw data.

I used DuckDB for SQL checks and pandas for a few charts/tables. I treated August as a partial month because the event data ends on 8 August 2026.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect()

## 1. Quick data checks

In [ ]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT borrower_id) AS unique_borrowers
FROM read_csv_auto('data/borrowers.csv')
""")

In [ ]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT payment_id) AS unique_payment_ids,
       COUNT(*) - COUNT(DISTINCT payment_id) AS duplicate_rows
FROM read_csv_auto('data/payments.csv')
""")

The borrower table has many repeated borrower IDs, so I should not use raw row count as the unique-borrower denominator. Payments also have duplicate payment IDs, which can inflate recovery.

## 2. Missing values

In [ ]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) - COUNT(borrower_id) AS missing_borrower_id,
       COUNT(*) - COUNT(principal_amount) AS missing_principal,
       COUNT(*) - COUNT(outstanding_amount) AS missing_outstanding,
       COUNT(*) - COUNT(dpd) AS missing_dpd,
       COUNT(*) - COUNT(risk_segment) AS missing_risk_segment,
       COUNT(*) - COUNT(status) AS missing_status
FROM read_csv_auto('data/accounts.csv')
""")

## 3. Clean payments before calculating recovery

In [ ]:
payments = con.sql("""
SELECT * FROM read_csv_auto('data/payments.csv')
QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY event_at DESC) = 1
""").df()
payments['event_at'] = pd.to_datetime(payments['event_at'])
payments.head()

I keep one record per payment_id. Most duplicate rows are exact copies; a small number differ only in a field such as payment_reference, so keeping the latest event is a simple and reproducible rule.

## 4. Recovery trend

In [ ]:
successful = payments[payments['payment_status'].eq('SUCCESS')].copy()
successful['month'] = successful['event_at'].dt.to_period('M').astype(str)
monthly = successful.groupby('month').agg(
    recovery=('amount','sum'),
    successful_payments=('payment_id','nunique'),
    accounts_paid=('account_id','nunique')
).reset_index()
monthly['mom_pct'] = monthly['recovery'].pct_change()*100
monthly['recovery_cr'] = monthly['recovery']/1e7
monthly

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(monthly['month'], monthly['recovery_cr'], marker='o')
plt.axhline(monthly['recovery_cr'].iloc[0], linestyle='--', linewidth=1)
plt.xticks(rotation=45)
plt.ylabel('Recovery (₹ Cr)')
plt.title('Monthly successful-payment recovery')
plt.tight_layout()
plt.show()

### What this says
The +11% figure appears for March versus February, but the trend is not consistently +11% month-on-month. April falls, June falls, and July rises again. August is incomplete and should not be compared with full months.

## 5. Segment checks

In [ ]:
accounts = pd.read_csv('data/accounts.csv')
successful_a = successful.merge(accounts[['account_id','risk_segment','dpd','loan_type']], on='account_id', how='left')
risk = successful_a.groupby('risk_segment').agg(recovery=('amount','sum'), accounts=('account_id','nunique')).reset_index()
risk['recovery_cr'] = risk['recovery']/1e7
risk['recovery_per_account'] = risk['recovery']/risk['accounts']
risk.sort_values('recovery', ascending=False)

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(risk['risk_segment'], risk['recovery_cr'])
plt.ylabel('Recovery (₹ Cr)')
plt.title('Recovery by risk segment')
plt.tight_layout()
plt.show()

## 6. Calls and operational context

In [ ]:
calls = pd.read_csv('data/calls.csv')
calls['event_at'] = pd.to_datetime(calls['event_at'])
calls['month'] = calls['event_at'].dt.to_period('M').astype(str)
call_month = calls.groupby('month').agg(
    calls=('call_id','count'),
    answered=('call_status', lambda x: (x=='ANSWERED').sum())
).reset_index()
call_month['contact_rate'] = call_month['answered']/call_month['calls']
call_month

## 7. Conclusion

**Fact:** recovery is volatile and the 11% month-on-month number is not sustained.

**Strong evidence:** duplicate payment IDs make raw payment sums unsafe; borrower duplicates also make raw borrower counts unsafe.

**Correlation:** segment and operational differences are useful for prioritisation, but the data is observational and does not prove causation.

**Recommendation:** do not commit the ₹10 Cr based on the current observational data alone. The strongest next step is a controlled targeting experiment with treatment/control groups and a pre-defined recovery window.